# PCA sur le dataset Réel (Shapes)

Ce notebook applique notre implémentation de PCA (`src/pca.py`) sur le jeu de données `Shapes` (composé d'images couleur de formes géométriques en 3x32x32, soit 3072 caractéristiques). Il génère également les métriques de compression et les visualisations Plotly associées.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("__file__"), "..")))

import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.dataset import load_shapes_dataset
from src.helper import extract_full_dataset
from src.pca import PCA
from src.metrics import compression_report, Latent

## 1. Chargement des données (Dataset Shapes)

In [2]:
# Activation de shuffle=True pour mélanger les classes (sinon, chargement alphabétique exclusif de 'bar')
dataloader = load_shapes_dataset(data_dir="../data/shapes_hard_color/train", batch_size=4096, shuffle=True)
images, shape_labels = extract_full_dataset(dataloader)

N = 10000  # Sous-ensemble pour garder une exécution fluide
X = images[:N].flatten(start_dim=1).numpy()
shape_labels = shape_labels[:N].numpy()

print("X shape:", X.shape)
print("Shape labels shape:", shape_labels.shape)

X shape: (10000, 3072)
Shape labels shape: (10000,)


## 2. Analyse de la variance expliquée cumulée

In [3]:
# On entraîne une PCA sur le nombre maximum de caractéristiques pour l'analyse de variance
pca_max = PCA(n_components=X.shape[1])
pca_max.fit(X)

total_variance = np.sum(pca_max.explained_variance)
explained_variance_ratio = pca_max.explained_variance / total_variance
cumulative_variance = np.cumsum(explained_variance_ratio)

fig = px.line(
    x=np.arange(1, len(cumulative_variance) + 1),
    y=cumulative_variance,
    labels=dict(x="Nombre de Composantes", y="Variance Expliquée Cumulée"),
    title="Courbe de Variance Expliquée Cumulée - Dataset Shapes"
)

thresholds = [0.90, 0.95, 0.99]
colors = ["red", "orange", "green"]
for t, color in zip(thresholds, colors):
    n_comp = np.where(cumulative_variance >= t)[0][0] + 1
    print(f"Seuil {t*100:.0f}% de variance atteint avec N = {n_comp} composantes.")
    fig.add_vline(x=n_comp, line_dash="dash", line_color=color)
    fig.add_annotation(x=n_comp, y=t, text=f"N={n_comp} ({t*100:.0f}%)", showarrow=True, arrowhead=1)

fig.update_layout(template="plotly_dark")
fig.show()

Seuil 90% de variance atteint avec N = 427 composantes.
Seuil 95% de variance atteint avec N = 711 composantes.
Seuil 99% de variance atteint avec N = 1104 composantes.


## 3. Fit avec le nombre de composantes choisi

In [4]:
# Nous choisissons N_COMPONENTS = 513 composantes pour capturer environ 90%
N_COMPONENTS = 513
model = PCA(n_components=N_COMPONENTS)
model.fit(X)

## 4. Encode / decode / rapport de compression

In [5]:
latent = model.encode(X)
X_reconstructed = model.decode(latent)
codebook = model.get_codebook()

report = compression_report(codebook, latent, X, X_reconstructed)
for k, v in report.items():
    print(f"{k}: {v}")

latent_nature: continuous
codebook_bytes: 6316032
latent_bytes: 20520000
total_compressed_bytes: 26836032
original_bytes: 122880000
compression_ratio: 4.57891837362543
reconstruction_mse: 0.0028860620222985744


## 5. Visualisation : les eigenvectors (composantes) comme images couleur

In [6]:
fig = make_subplots(
    rows=2, cols=5,
    subplot_titles=[f"Eigenvector {i+1}" for i in range(10)]
)

for i in range(10):
    row = i // 5 + 1
    col = i % 5 + 1
    
    # model.components est de forme (3072, n_components)
    # Reshape en (3, 32, 32) puis permutation en (32, 32, 3) pour l'affichage RGB
    comp_vector = model.components[:, i].reshape(3, 32, 32).transpose(1, 2, 0)
    
    # Normalisation des valeurs dans [0, 1] pour affichage correct des canaux de couleur
    c_min, c_max = comp_vector.min(), comp_vector.max()
    comp_img = (comp_vector - c_min) / (c_max - c_min)
    
    fig.add_trace(
        go.Image(z=(comp_img * 255).astype(np.uint8)),
        row=row, col=col
    )

fig.update_layout(
    height=500, width=1000,
    title_text="Eigenvectors du Dataset Shapes (Vecteurs de Variation)",
    template="plotly_dark"
)

for i in range(1, 11):
    fig.update_yaxes(showticklabels=False, row=(i-1)//5+1, col=(i-1)%5+1)
    fig.update_xaxes(showticklabels=False, row=(i-1)//5+1, col=(i-1)%5+1)

fig.show()

## 6. Visualisation : images originales vs reconstruites

In [7]:
np.random.seed(42)
indices = np.random.choice(X.shape[0], size=5, replace=False)

fig = make_subplots(
    rows=2, cols=5,
    subplot_titles=[f"Original {idx}" for idx in indices] + [f"Reconstructed {idx}" for idx in indices]
)

for i, idx in enumerate(indices):
    orig_img = X[idx].reshape(3, 32, 32).transpose(1, 2, 0)
    recon_img = X_reconstructed[idx].reshape(3, 32, 32).transpose(1, 2, 0)
    
    # Écrêtage des valeurs dans l'intervalle [0, 1]
    orig_img = np.clip(orig_img, 0.0, 1.0)
    recon_img = np.clip(recon_img, 0.0, 1.0)
    
    fig.add_trace(go.Image(z=(orig_img * 255).astype(np.uint8)), row=1, col=i+1)
    fig.add_trace(go.Image(z=(recon_img * 255).astype(np.uint8)), row=2, col=i+1)

fig.update_layout(
    height=500, width=1000,
    title_text="Reconstructions PCA (Original vs Reconstruit)",
    template="plotly_dark"
)

for row in [1, 2]:
    for col in range(1, 6):
        fig.update_yaxes(showticklabels=False, row=row, col=col)
        fig.update_xaxes(showticklabels=False, row=row, col=col)

fig.show()

## 7. Visualisation : projection 2D de l'espace latent

In [8]:
pca_2d = PCA(n_components=2)
pca_2d.fit(X)

# Mélange l'ordre des points pour éviter l'effet de couches de tracé
perm = np.random.default_rng(42).permutation(X.shape[0])
X_plot = X[perm]
shape_labels_plot = shape_labels[perm]
latent_2d = pca_2d.encode(X_plot)

# Noms des 6 classes géométriques
class_names = ["bar", "circle", "cross", "square", "star", "triangle"]
shape_names = [class_names[lbl] for lbl in shape_labels_plot]

fig = px.scatter(
    x=latent_2d.array[:, 0],
    y=latent_2d.array[:, 1],
    color=shape_names,
    color_discrete_sequence=px.colors.qualitative.Plotly,
    labels=dict(x="Premiere Composante Principale", y="Deuxieme Composante Principale", color="Forme Géométrique"),
    title="Projection dans l'espace latent 2D de la PCA",
    opacity=0.6,
    render_mode="webgl"
)

fig.update_traces(marker=dict(size=4, line=dict(width=0)))
fig.update_layout(
    width=900, height=600,
    template="plotly_dark"
)
fig.show()

## 8. Génération de nouvelles images par sampling dans l'espace latent

Ici encore, on génère un nouveau point latent plausible en restant dans une zone bien peuplée du dataset.

In [9]:
latent_train = latent.array
latent_mean = latent_train.mean(axis=0)
latent_std = latent_train.std(axis=0)

rng = np.random.default_rng(123)

# Approche 1 : point latent proche d'un point d'entraînement existant
z_ref = latent_train[rng.integers(len(latent_train))]
z_new_dense = z_ref + rng.normal(scale=0.2 * latent_std, size=z_ref.shape)

# Approche 2 : choix uniforme dans le bounding box latent puis vérification
latent_min = latent_train.min(axis=0)
latent_max = latent_train.max(axis=0)
z_new_box = rng.uniform(latent_min, latent_max)
box_dist = np.linalg.norm(latent_train - z_new_box, axis=1)
print(f"Distance minimale au dataset latent (box) : {box_dist.min():.4f}")

X_new_dense = model.decode(Latent(array=z_new_dense[None, :], nature="continuous"))
X_new_box = model.decode(Latent(array=z_new_box[None, :], nature="continuous"))

# Reshape et clipping pour l'affichage couleur
images = []
for X_gen in [X_new_dense, X_new_box]:
    img = X_gen[0].reshape(3, 32, 32).transpose(1, 2, 0)
    img = np.clip(img, 0.0, 1.0)
    images.append((img * 255).astype(np.uint8))

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "Sampling autour d'un point dense",
    "Sampling dans un bounding box latent"
])

for col, img in enumerate(images, start=1):
    fig.add_trace(go.Image(z=img), row=1, col=col)

fig.update_layout(
    height=400, width=800,
    title_text="Nouvelles images Shapes générées par sampling latent",
    template="plotly_dark"
)
for col in [1, 2]:
    fig.update_yaxes(showticklabels=False, row=1, col=col)
    fig.update_xaxes(showticklabels=False, row=1, col=col)
fig.show()

Distance minimale au dataset latent (box) : 21.7590


## 9. Reconstruction pour plusieurs valeurs de k

On compare la reconstruction pour différentes valeurs de `n_components` (k). Sur ce dataset couleur (3072 features), il faut k≈513 composantes pour expliquer 90% de la variance. On observe la progression depuis k=2 (quasi constant) jusqu'à k=800 (quasi parfait).

In [10]:
K_VALUES = [2, 10, 50, 100, 200, 513, 800]

np.random.seed(42)
sample_indices = np.random.choice(X.shape[0], size=5, replace=False)

# Pré-calculer les reconstructions pour chaque k
reconstructions = {}
for k in K_VALUES:
    pca_k = PCA(n_components=k)
    pca_k.fit(X)
    latent_k = pca_k.encode(X)
    recon_k = pca_k.decode(latent_k)
    reconstructions[k] = recon_k
    print(f"k={k:>4d} → MSE = {np.mean((X - recon_k)**2):.4f}")

k=   2 → MSE = 0.0265
k=  10 → MSE = 0.0161
k=  50 → MSE = 0.0097
k= 100 → MSE = 0.0077
k= 200 → MSE = 0.0058
k= 513 → MSE = 0.0029
k= 800 → MSE = 0.0013


In [11]:
n_rows = 1 + len(K_VALUES)  # 1 row for originals + 1 per k
n_cols = 5

row_titles = ["Original"] + [f"k = {k}" for k in K_VALUES]

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    vertical_spacing=0.02,
    horizontal_spacing=0.02,
    row_titles=row_titles
)

for i, idx in enumerate(sample_indices):
    # Ligne 1 : original
    orig_img = np.clip(X[idx].reshape(3, 32, 32).transpose(1, 2, 0), 0, 1)
    fig.add_trace(go.Image(z=(orig_img * 255).astype(np.uint8)), row=1, col=i+1)

    # Lignes suivantes : reconstructions pour chaque k
    for j, k in enumerate(K_VALUES):
        recon_img = np.clip(reconstructions[k][idx].reshape(3, 32, 32).transpose(1, 2, 0), 0, 1)
        fig.add_trace(go.Image(z=(recon_img * 255).astype(np.uint8)), row=j+2, col=i+1)

fig.update_layout(
    height=200 * n_rows,
    width=1000,
    title_text="Reconstruction PCA pour différentes valeurs de k (Shapes)",
    template="plotly_dark",
    showlegend=False
)

for r in range(1, n_rows + 1):
    for c in range(1, n_cols + 1):
        fig.update_yaxes(showticklabels=False, row=r, col=c)
        fig.update_xaxes(showticklabels=False, row=r, col=c)

fig.show()